# 10. Fine-tuning V1

Этот ноутбук сохраняет проверенную реализацию исходного эксперимента и имена его
артефактов. Запускайте **Restart Kernel and Run All Cells** после выполнения всех
предыдущих пронумерованных ноутбуков.

Все входы, кроме исходных raw-данных из `config/raw_sources.json`, создаются внутри
этого проекта. Результаты записываются в `outputs/`, а модели — в `checkpoints/`.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Неверный.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from project_paths import *
ensure_project_directories()

print("Project root:", PROJECT_ROOT)

Project root: D:\Users\user\Desktop\DS_XRD_project


# Fine-tuning голов XRD на реальных данных — v1

**Данные**: 3 301 train / 174 val реальных спектров (RRUFF + opXRD, сплит 95/5 из `splits_ft.parquet`)
+ replay синтетики 50/50 в каждом батче (против катастрофического забывания).

**Старт**: чекпойнт пре-трейна `pretrain_v1_full_best.pt` (460k синтетики, 12 эпох).

**Ключевые решения**:
- λ-conditioning: 180 opXRD-строк без λ → импутация Cu Kα1 (1.5406);
- головы с масками: у RRUFF нет SG — маска обнуляет лосс автоматически;
- дифференциальный LR: бэкбон 1e-5, головы 3e-5;
- метрики до/после FT на одном и том же реальном вале.

**Режимы** (MODE): `test` — 3 эпохи, быстрая проверка; `full` — 25 эпох с early best.

In [2]:
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

device: cuda NVIDIA GeForce RTX 3060 Ti


In [3]:
MODE = 'full'   # 'test' | 'full'

BASE = PROJECT_ROOT
DATA = BASE / 'data' / 'preprocessed'
CKPT_PRE = BASE / 'checkpoints' / 'pretrain_v1_full_best.pt'
OUT = BASE / 'outputs'
CKPT_DIR = BASE / 'checkpoints'

GRID_N = 4096
SYSTEMS = ['triclinic', 'monoclinic', 'orthorhombic', 'tetragonal',
           'trigonal', 'hexagonal', 'cubic']
IMPUTE_LAMBDA = 1.5406   # Cu Ka1 для строк без λ

HP = dict(
    batch=128,
    lr_head=3e-5,
    lr_backbone=1e-5,
    wd=1e-4,
    clip=1.0,
    epochs={'test': 3, 'full': 25}[MODE],
    replay_frac=0.5,          # доля синтетики в батче
    warmup_frac=0.1,
)
print('MODE =', MODE, '| HP:', HP)

MODE = full | HP: {'batch': 128, 'lr_head': 3e-05, 'lr_backbone': 1e-05, 'wd': 0.0001, 'clip': 1.0, 'epochs': 25, 'replay_frac': 0.5, 'warmup_frac': 0.1}


In [ ]:
# ---------- общие артефакты пре-трейна ----------
stats = json.loads((OUT / 'pretrain_stats.json').read_text())
VOCAB = stats['vocab']
EL_IDX = {e: i for i, e in enumerate(VOCAB)}
LAT_MEAN = np.array(stats['lat_mean'])
LAT_STD = np.array(stats['lat_std'])
VOL_MEAN, VOL_STD = stats['vol_mean'], stats['vol_std']
print('словарь элементов:', len(VOCAB))

index = pd.read_parquet(DATA / 'index_preprocessed.parquet')
N_TOTAL = len(index)
X_MM = np.memmap(DATA / 'X_intensity.f16', dtype=np.float16, mode='r', shape=(N_TOTAL, GRID_N))
M_MM = np.memmap(DATA / 'M_mask.u8', dtype=np.uint8, mode='r', shape=(N_TOTAL, GRID_N))
row_of = dict(zip(index['sample_id'], index['row_idx']))

# ---------- реальный FT-пул ----------
ft = pd.read_parquet(BASE / 'data' / 'clean' / 'ft_pool_combined.parquet')
sp = pd.read_parquet(OUT / 'splits_ft.parquet')[['sample_id', 'ft_split']]
ft = ft.merge(sp, on='sample_id')
ft['row_idx'] = ft['sample_id'].map(row_of)
assert ft['row_idx'].notna().all(), 'не все FT-строки найдены в индексе этапа B'

# импутация λ
n_imp = int(ft['primary_wavelength'].isna().sum())
ft['primary_wavelength'] = ft['primary_wavelength'].fillna(IMPUTE_LAMBDA)
ft['secondary_wavelength'] = ft['secondary_wavelength'].fillna(np.nan)
print(f'FT-пул: {len(ft)} строк, λ импутировано {n_imp}')

# метки решётки/объёма
abc = ft[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang = ft[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
ft['V'] = abc.prod(1) * np.sqrt(np.clip(t, 1e-12, None))
ft['lat6'] = list(np.hstack([np.log(abc), ang]))

def to_list(v):

    if isinstance(v, str):
        try:
            return json.loads(v)
        
        except Exception:
            return []
        
    if isinstance(v, (list, tuple, np.ndarray)):
        return [e for e in v if isinstance(e, str)]
    
    return []

ft['elements'] = ft['elements_list'].apply(to_list)

tr_real = ft[ft['ft_split'] == 'train'].reset_index(drop=True)
va_real = ft[ft['ft_split'] == 'val'].reset_index(drop=True)
print(f'real: train {len(tr_real)} / val {len(va_real)}')
print(va_real.groupby('dataset_role').size().to_string())

# ---------- replay-пул синтетики (train-часть пре-трейна) ----------

splits_pre = pd.read_parquet(OUT / 'splits_pretrain.parquet')
syn_meta = pd.read_parquet(BASE / 'data' / 'clean' / 'df_synth_summary_final_clean.parquet')[
    ['sample_id', 'lattice_a', 'lattice_b', 'lattice_c',
     'alpha', 'beta', 'gamma', 'spacegroup_number', 'crystal_system', 'elements_list']
]
pre_rows = index[index['split_role'] == 'pretrain'][['sample_id', 'row_idx',
                                                      'lambda_1', 'lambda_2']]
syn = pre_rows.merge(splits_pre, on='sample_id').merge(syn_meta, on='sample_id')
syn = syn[syn['split'] == 'train'].reset_index(drop=True)  # только трейн синтетики!
abc_s = syn[['lattice_a', 'lattice_b', 'lattice_c']].to_numpy(np.float64)
ang_s = syn[['alpha', 'beta', 'gamma']].to_numpy(np.float64)
ca, cb, cg = (np.cos(np.radians(ang_s[:, i])) for i in range(3))
t = 1.0 - ca**2 - cb**2 - cg**2 + 2 * ca * cb * cg
syn['V'] = abc_s.prod(1) * np.sqrt(np.clip(t, 1e-12, None))
syn['lat6'] = list(np.hstack([np.log(abc_s), ang_s]))
syn['elements'] = syn['elements_list'].apply(to_list)

print('replay-пул синтетики (train):', len(syn))

словарь элементов: 98
FT-пул: 3475 строк, λ импутировано 180
real: train 3301 / val 174
dataset_role
opxrd    106
rruff     68
replay-пул синтетики (train): 451027


In [ ]:
# ---------- датасеты ----------

def build_labels(frame, is_real):

    lam1 = frame['lambda_1' if 'lambda_1' in frame else 'primary_wavelength']
    lam1 = lam1.to_numpy(np.float32)
    lam2 = (frame['lambda_2'] if 'lambda_2' in frame
            else frame['secondary_wavelength']).to_numpy(np.float32)
    
    lam = np.stack([lam1 / 1.54, np.nan_to_num(lam2) / 1.54,
                    np.isfinite(lam2).astype(np.float32)], 1)

    latm = frame[['lattice_a']].notna().all(1).to_numpy(np.float32) if is_real \
        else np.ones(len(frame), np.float32)
    
    has_lat = ~np.isnan(np.stack(frame['lat6'].to_numpy())).any(1)
    latm = has_lat.astype(np.float32)
    lat6 = np.stack(frame['lat6'].to_numpy())
    lat6 = (lat6 - LAT_MEAN) / LAT_STD

    vol = (np.log(frame['V'].to_numpy(np.float64)) - VOL_MEAN) / VOL_STD

    sg_raw = frame['spacegroup_number'].to_numpy(float)
    sgm = np.isfinite(sg_raw).astype(np.float32)
    sg = np.nan_to_num(sg_raw).astype(np.int64) - 1

    sysmap = {s: i for i, s in enumerate(SYSTEMS)}
    sys_raw = frame['crystal_system'].map(sysmap)
    sysm = sys_raw.notna().to_numpy(np.float32)
    sys_ = sys_raw.fillna(0).to_numpy(np.int64)

    n = len(frame)
    el = np.zeros((n, len(VOCAB)), np.float32)
    elm = np.zeros(n, np.float32)

    for i, els in enumerate(frame['elements']):
        if len(els):
            elm[i] = 1.0
            for e in els:
                j = EL_IDX.get(e)
                if j is not None:
                    el[i, j] = 1.0

    return dict(lam=lam, lat6=lat6.astype(np.float32), latm=latm,
                vol=vol.astype(np.float32), volm=latm.copy(),
                sg=sg, sgm=sgm, sys_=sys_, sysm=sysm, el=el, elm=elm)

class SpecDS(Dataset):

    """Спектры + метки; row_idx ссылается на меммапы этапа B."""

    def __init__(self, frame, is_real):
        self.row = frame['row_idx'].to_numpy(np.int64)
        self.role = frame['dataset_role'].to_numpy() if is_real else np.array(['synth'] * len(frame))
        self.L = build_labels(frame, is_real)

    def __len__(self):
        return len(self.row)

    def __getitem__(self, i):
        x = np.empty((2, GRID_N), np.float32)
        x[0] = X_MM[self.row[i]]
        x[1] = M_MM[self.row[i]]
        L = self.L

        return (torch.from_numpy(x), torch.from_numpy(L['lam'][i]),
                torch.from_numpy(L['lat6'][i]), torch.tensor(L['latm'][i]),
                torch.tensor(L['sg'][i]), torch.tensor(L['sgm'][i]),
                torch.tensor(L['sys_'][i]), torch.tensor(L['sysm'][i]),
                torch.from_numpy(L['el'][i]), torch.tensor(L['elm'][i]),
                torch.tensor(L['vol'][i]), torch.tensor(L['volm'][i]))

class MixedLoader:

    """Каждый батч: (1-replay_frac) реальных + replay_frac синтетики."""

    def __init__(self, real_ds, syn_ds, batch, frac):
        self.real = real_ds
        self.syn = syn_ds
        self.batch = batch
        self.frac = frac
        self.n_syn_per_batch = max(1, int(batch * frac))
        self.n_real_per_batch = max(1, batch - self.n_syn_per_batch)
        self.steps = len(real_ds) // self.n_real_per_batch
        self._syn_iter = None
        self._syn_left = []

    def __len__(self):
        return self.steps

    def __iter__(self):
        real_idx = np.random.permutation(len(self.real))
        syn_idx = np.random.permutation(len(self.syn))
        sp = 0

        for b in range(self.steps):
            r_idx = real_idx[b * self.n_real_per_batch:(b + 1) * self.n_real_per_batch]
            s_idx = syn_idx[(b * self.n_syn_per_batch) % len(syn_idx):
                            (b * self.n_syn_per_batch) % len(syn_idx) + self.n_syn_per_batch]
            items = [self.real[i] for i in r_idx] + [self.syn[j] for j in s_idx]
            yield [torch.stack(t) for t in zip(*items)]

val_loader = DataLoader(SpecDS(va_real, True), batch_size=HP['batch'],
                        shuffle=False, pin_memory=True)

print('val batches:', len(val_loader))

val batches: 2


In [ ]:
# ---------- модель (архитектура пре-трейна) + загрузка чекпойнта ----------
class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.n1 = nn.GroupNorm(8, cout)
        self.conv2 = nn.Conv1d(cout, cout, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, cout)

        if cin == cout and stride == 1:
            self.skip = nn.Identity()

        else:
            self.skip = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False),
                                      nn.GroupNorm(8, cout))

    def forward(self, x):
        h = F.gelu(self.n1(self.conv1(x)))
        h = self.n2(self.conv2(h))

        return F.gelu(h + self.skip(x))

class XRDNet(nn.Module):
    def __init__(self, n_el):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(2, 32, 15, padding=7, bias=False),
                                  nn.GroupNorm(8, 32), nn.GELU())
        chans = [(32, 48), (48, 64), (64, 96), (96, 128), (128, 192), (192, 256)]
        self.blocks = nn.Sequential(*[ResBlock(ci, co, stride=2) for ci, co in chans])
        self.lam_mlp = nn.Sequential(nn.Linear(3, 16), nn.GELU(), nn.Linear(16, 16))
        self.trunk = nn.Sequential(nn.Linear(256 + 16, 512), nn.GELU(),
                                   nn.Linear(512, 512), nn.GELU())
        self.head_lat = nn.Linear(512, 6)
        self.head_vol = nn.Linear(512, 1)
        self.head_sg = nn.Linear(512, 230)
        self.head_sys = nn.Linear(512, 7)
        self.head_el = nn.Linear(512, n_el)

    def forward(self, x, lam):
        f = self.stem(x)
        f = self.blocks(f)
        w = F.adaptive_avg_pool1d(x[:, 1:2], f.shape[-1]).clamp_min(1e-3)
        pooled = (f * w).sum(-1) / w.sum(-1)
        z = torch.cat([pooled, self.lam_mlp(lam)], dim=1)
        z = self.trunk(z)

        return dict(lat=self.head_lat(z), vol=self.head_vol(z).squeeze(-1),
                    sg=self.head_sg(z), sys=self.head_sys(z), el=self.head_el(z))

model = XRDNet(len(VOCAB)).to(DEVICE)
sd = torch.load(CKPT_PRE, map_location=DEVICE, weights_only=True)
model.load_state_dict(sd)

print('чекпойнт пре-трейна загружен:', CKPT_PRE.name)
print('параметров: %.2fM' % (sum(p.numel() for p in model.parameters()) / 1e6))

чекпойнт пре-трейна загружен: pretrain_v1_full_best.pt
параметров: 1.37M


In [ ]:
# ---------- лоссы и метрики ----------

def masked_l1(pred, tgt, mask):
    m = mask > 0

    if m.sum() == 0:
        return pred.new_zeros(())
    
    return F.smooth_l1_loss(pred[m], tgt[m])

def masked_ce(logits, tgt, mask):
    m = mask > 0

    if m.sum() == 0:
        return logits.new_zeros(())
    
    return F.cross_entropy(logits[m], tgt[m].long())

def compute_losses(out, batch):
    (_, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm) = batch
    # BCE только по размеченным строкам (mask elm)

    if elm.sum() > 0:
        m_el = (elm > 0).unsqueeze(1)
        loss_el = F.binary_cross_entropy_with_logits(
            out['el'][m_el.squeeze(1)], el[m_el.squeeze(1)])
        
    else:
        loss_el = out['el'].new_zeros(())

    return dict(
        lat=masked_l1(out['lat'], lat, latm),
        vol=masked_l1(out['vol'], vol, volm),
        sg=masked_ce(out['sg'], sg, sgm),
        sys=masked_ce(out['sys'], sys_, sysm),
        el=loss_el,
    )

@torch.no_grad()
def evaluate(model, loader, tag=''):
    model.eval()
    rows = []

    for batch in loader:
        batch = [b.to(DEVICE, non_blocking=True) for b in batch]
        x, lam = batch[0], batch[1]

        with torch.autocast('cuda', dtype=torch.float16):
            out = model(x, lam)
        _, _, lat, latm, sg, sgm, sys_, sysm, el, elm, vol, volm = batch
        lat_pred = out['lat'].float().cpu().numpy() * LAT_STD + LAT_MEAN
        lat_pred[:, :3] = np.exp(lat_pred[:, :3])
        lat_true = lat.cpu().numpy() * LAT_STD + LAT_MEAN
        lat_true[:, :3] = np.exp(lat_true[:, :3])

        for i in range(len(x)):
            rows.append(dict(
                latm=float(latm[i]), sgm=float(sgm[i]), sysm=float(sysm[i]),
                elm=float(elm[i]),
                sg_ok=float(sgm[i] > 0 and out['sg'][i].argmax().item() == sg[i].item()),
                sg_top5=float(sgm[i] > 0 and sg[i].item() in out['sg'][i].topk(5).indices.tolist()),
                sys_ok=float(sysm[i] > 0 and out['sys'][i].argmax().item() == sys_[i].item()),
                mae_a=abs(lat_pred[i, 0] - lat_true[i, 0]) if latm[i] > 0 else np.nan,
                mae_ang=float(np.abs(lat_pred[i, 3:] - lat_true[i, 3:]).mean()) if latm[i] > 0 else np.nan,
                el_ok=float(elm[i] > 0 and (set(np.where(el[i].cpu().numpy() > 0.5)[0]) ==
                                            set(np.where(el[i].cpu().numpy() > 0.5)[0]))),
                pred_el=set(np.where(torch.sigmoid(out['el'][i]).cpu().numpy() > 0.5)[0]),
                true_el=set(np.where(el[i].cpu().numpy() > 0.5)[0]) if elm[i] > 0 else set(),
            ))
    model.train()
    d = pd.DataFrame(rows)
    res = {}
    v = d[d['sgm'] > 0]
    res['sg_acc'] = v['sg_ok'].mean() if len(v) else np.nan
    res['sg_top5'] = v['sg_top5'].mean() if len(v) else np.nan
    v = d[d['sysm'] > 0]
    res['sys_acc'] = v['sys_ok'].mean() if len(v) else np.nan
    v = d[d['latm'] > 0]

    if len(v):
        res['mae_a'] = v['mae_a'].mean()
        res['mae_a_med'] = v['mae_a'].median()
        res['mae_ang'] = v['mae_ang'].mean()
    v = d[d['elm'] > 0]

    if len(v):
        tp = sum(len(r['pred_el'] & r['true_el']) for _, r in v.iterrows())
        fp = sum(len(r['pred_el'] - r['true_el']) for _, r in v.iterrows())
        fn = sum(len(r['true_el'] - r['pred_el']) for _, r in v.iterrows())
        p = tp / max(tp + fp, 1)
        rc = tp / max(tp + fn, 1)
        res['el_f1_micro'] = 2 * p * rc / max(p + rc, 1e-9)
        res['el_exact'] = float((v['pred_el'] == v['true_el']).mean())
    res['n'] = len(d)
    
    return res

def fmt(m):
    return {k: (round(float(v), 4) if isinstance(v, (int, float, np.floating))
                and not isinstance(v, bool) and v == v else v)
            for k, v in m.items()}

In [ ]:
# ---------- zero-shot: пре-трейн модель на РЕАЛЬНОМ вале (до FT) ----------

zero_shot = evaluate(model, val_loader)

print('ZERO-SHOT (пре-трейн, реальный val, n=%d):' % zero_shot['n'])
print(json.dumps(fmt(zero_shot), indent=2, ensure_ascii=False))

ZERO-SHOT (пре-трейн, реальный val, n=174):
{
  "sg_acc": 0.0,
  "sg_top5": 0.0345,
  "sys_acc": 0.3684,
  "mae_a": 4.3114,
  "mae_a_med": 2.3207,
  "mae_ang": 5.3233,
  "el_f1_micro": 0.1511,
  "el_exact": 0.0,
  "n": 174.0
}


In [ ]:
# ---------- fine-tuning ----------
backbone_params = [p for n, p in model.named_parameters()
                   if not n.startswith(('head_', 'trunk', 'lam_mlp'))]
head_params = [p for n, p in model.named_parameters()
               if n.startswith(('head_', 'trunk', 'lam_mlp'))]
opt = torch.optim.AdamW([
    {'params': backbone_params, 'lr': HP['lr_backbone']},
    {'params': head_params, 'lr': HP['lr_head']},
], weight_decay=HP['wd'])
scaler = torch.amp.GradScaler('cuda')

real_ds = SpecDS(tr_real, True)
syn_ds = SpecDS(syn, False)
loader = MixedLoader(real_ds, syn_ds, HP['batch'], HP['replay_frac'])
steps_total = HP['epochs'] * len(loader)
warmup = max(1, int(steps_total * HP['warmup_frac']))

def lr_scale(step):
    if step < warmup:
        return step / warmup
    
    p = (step - warmup) / max(steps_total - warmup, 1)

    return 0.5 * (1.0 + math.cos(math.pi * min(p, 1.0)))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_scale)

best_metric = -1.0
history = []
t0 = time.time()
step = 0

for epoch in range(HP['epochs']):
    runma = defaultdict(float)
    cnt = 0
    for batch in loader:
        batch = [b.to(DEVICE, non_blocking=True) for b in batch]

        with torch.autocast('cuda', dtype=torch.float16):
            out = model(batch[0], batch[1])
            losses = compute_losses(out, batch)
            total = sum(losses.values())

        opt.zero_grad(set_to_none=True)
        scaler.scale(total).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), HP['clip'])
        scaler.step(opt)
        scaler.update()
        sched.step()
        step += 1
        cnt += 1

        for k, v in losses.items():
            runma[k] += float(v)

        runma['total'] += float(total)
    m = evaluate(model, val_loader)
    score = (m.get('sys_acc') or 0) + (m.get('el_f1_micro') or 0) \
            + (m.get('sg_acc') or 0) * 0.5
    history.append(dict(epoch=epoch + 1, **{k: runma[k] / cnt for k in
                                             ['total', 'lat', 'sg', 'sys', 'el', 'vol']}, **m))
    print(f"epoch {epoch+1:2d}/{HP['epochs']} | loss {runma['total']/cnt:.3f} | "
          f"sys {m.get('sys_acc', float('nan')):.3f} sg {m.get('sg_acc', float('nan')):.3f} "
          f"el_f1 {m.get('el_f1_micro', float('nan')):.3f} "
          f"mae_a {m.get('mae_a', float('nan')):.2f} "
          f"| {time.time()-t0:.0f} c", flush=True)
    
    if score > best_metric:
        best_metric = score
        torch.save(model.state_dict(), CKPT_DIR / 'ft_v1_best.pt')

torch.save(model.state_dict(), CKPT_DIR / 'ft_v1_last.pt')

print(f'готово за {(time.time()-t0)/60:.1f} мин')

d:\Dev\envs\study\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


epoch  1/25 | loss 4.871 | sys 0.436 sg 0.000 el_f1 0.136 mae_a 4.15 | 5 c
epoch  2/25 | loss 4.434 | sys 0.481 sg 0.103 el_f1 0.173 mae_a 4.02 | 9 c
epoch  3/25 | loss 4.300 | sys 0.489 sg 0.138 el_f1 0.211 mae_a 3.94 | 13 c
epoch  4/25 | loss 4.075 | sys 0.534 sg 0.207 el_f1 0.228 mae_a 3.90 | 17 c
epoch  5/25 | loss 4.000 | sys 0.526 sg 0.276 el_f1 0.239 mae_a 3.83 | 21 c
epoch  6/25 | loss 3.851 | sys 0.549 sg 0.552 el_f1 0.262 mae_a 3.77 | 25 c
epoch  7/25 | loss 3.852 | sys 0.534 sg 0.655 el_f1 0.261 mae_a 3.75 | 29 c
epoch  8/25 | loss 3.776 | sys 0.541 sg 0.793 el_f1 0.277 mae_a 3.72 | 32 c
epoch  9/25 | loss 3.721 | sys 0.526 sg 0.897 el_f1 0.287 mae_a 3.72 | 36 c
epoch 10/25 | loss 3.691 | sys 0.526 sg 0.897 el_f1 0.316 mae_a 3.68 | 40 c
epoch 11/25 | loss 3.680 | sys 0.549 sg 0.897 el_f1 0.333 mae_a 3.64 | 44 c
epoch 12/25 | loss 3.682 | sys 0.556 sg 0.897 el_f1 0.341 mae_a 3.64 | 48 c
epoch 13/25 | loss 3.663 | sys 0.541 sg 0.897 el_f1 0.351 mae_a 3.61 | 52 c
epoch 14/25 | 

In [ ]:
# ---------- итоговое сравнение: zero-shot vs FT ----------
model.load_state_dict(torch.load(CKPT_DIR / 'ft_v1_best.pt', map_location=DEVICE,
                                 weights_only=True))
after = evaluate(model, val_loader)

print('REAL VAL (n=%d): zero-shot пре-трейна -> после FT' % after['n'])
keys = ['sys_acc', 'sg_acc', 'sg_top5', 'el_f1_micro', 'el_exact',
        'mae_a', 'mae_a_med', 'mae_ang']
cmp = pd.DataFrame({
    'zero_shot': [zero_shot.get(k) for k in keys],
    'after_ft': [after.get(k) for k in keys],
}, index=keys)
print(cmp.round(4).to_string())

h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(h['epoch'], h['total'], marker='o')
axes[0].set_title('train loss'); axes[0].set_yscale('log')

for k, ax in [('sys_acc', axes[1]), ('el_f1_micro', axes[2])]:
    ax.plot(h['epoch'], h[k], marker='o', label=k)
    ax.plot(h['epoch'], h['mae_a'] / 10 if k == 'el_f1_micro' else h[k] * 0 + np.nan,
            alpha=0)  # placeholder
    ax.set_ylim(0, 1); ax.legend()

if 'mae_a' in h:
    axes[2].plot(h['epoch'], (h['mae_a'].max() - h['mae_a']) /
                 max(h['mae_a'].max() - h['mae_a'].min(), 1e-9),
                 marker='.', ls='--', label='mae_a (норм.)')
    axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)
    
plt.tight_layout()
plt.savefig(OUT / f'ft_{MODE}_metrics.png', dpi=130)
print('график:', OUT / f'ft_{MODE}_metrics.png')

REAL VAL (n=174): zero-shot пре-трейна -> после FT
             zero_shot  after_ft
sys_acc         0.3684    0.5414
sg_acc          0.0000    0.8966
sg_top5         0.0345    1.0000
el_f1_micro     0.1511    0.3964
el_exact        0.0000    0.1504
mae_a           4.3114    3.5486
mae_a_med       2.3207    1.7678
mae_ang         5.3233    5.2439
график: D:\Users\user\Desktop\DS_XRD_project\outputs\ft_full_metrics.png
